In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import login

login(os.getenv('HF_TOKEN'))

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading model... (3-5 min)")
base = AutoModelForCausalLM.from_pretrained(
    'mistralai/Mistral-7B-Instruct-v0.3',
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model     = PeftModel.from_pretrained(base, 'sachinwebiwork/ai-interviewer-mistral')
tokenizer = AutoTokenizer.from_pretrained('sachinwebiwork/ai-interviewer-mistral')
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'
model.eval()
print("✅ Model ready!")

In [ ]:
# ════════════════════════════════════════════════════════════
# AI INTERVIEWER — SIMPLE INPUT-BASED (NO WIDGETS)
# Run this ONE cell — answers Python input() box mein doge
# ════════════════════════════════════════════════════════════

import torch, re, json

# ── CONFIG ────────────────────────────────────────────────
NUM_QUESTIONS = 5
ROLE          = "Gen AI Developer"
EXPERIENCE    = "2+ Years"

RESUME_SKILLS = [
    "LangChain","LangGraph","LangSmith","Azure OpenAI",
    "RAG","Pinecone","ChromaDB","pgvector",
    "Multi-Agent Systems","Prompt Engineering","FastAPI","Python","Docker"
]
RESUME_PROJECTS = [
    "Enterprise Knowledge Platform: LangGraph, Azure OpenAI, ChromaDB, RAG, 45% accuracy improvement",
    "AI Customer Support: LangGraph agents, Pinecone, 50% automated resolution",
    "Candidate Evaluation: multi-agent LangGraph, 60% less manual screening",
]
JD_SKILLS  = ["LangChain","LangGraph","RAG","vector databases","Python","LLMs","Azure OpenAI","Docker"]
JD_COMPANY = "Remote AI Startup"

# ── STATE ─────────────────────────────────────────────────
state = {"questions": [], "answers": [], "feedbacks": [], "scores": [], "actions": []}

# ── PROMPT BUILDER ────────────────────────────────────────
def build_prompt(user_turn):
    system = (
        f"You are a strict technical interviewer.\n"
        f"Role: {ROLE} | Experience: {EXPERIENCE}\n"
        f"Company: {JD_COMPANY}\n"
        f"JD Skills: {', '.join(JD_SKILLS)}\n"
        f"Resume Skills: {', '.join(RESUME_SKILLS[:6])}\n"
        f"Resume Project: {RESUME_PROJECTS[0]}\n"
    )
    return f"<s>[INST] {system}\n\n{user_turn} [/INST]"

# ── FINE-TUNED MODEL — Question + Feedback/Score/Action ───
def run_model(prompt_text, max_tok=200):
    inputs = tokenizer(prompt_text, return_tensors='pt', truncation=True, max_length=2048).to('cuda')
    with torch.no_grad():
        out = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_new_tokens=max_tok,
            min_new_tokens=20,
            do_sample=True,
            temperature=0.8,
            top_p=0.92,
            top_k=50,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def run_feedback_model(prompt_text, max_tok=150):
    inputs = tokenizer(prompt_text, return_tensors='pt', truncation=True, max_length=2048).to('cuda')
    with torch.no_grad():
        out = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_new_tokens=max_tok,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

# ── BASE MODEL — Final Report (metadata not trained, so use base) ──
def run_base_model(prompt_text, max_tok=400):
    """Use underlying base Mistral (no LoRA) for final hiring report."""
    base_model = model.base_model  # access base model inside PeftModel
    inputs = tokenizer(prompt_text, return_tensors='pt', truncation=True, max_length=2048).to('cuda')
    with torch.no_grad():
        out = base_model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_new_tokens=max_tok,
            min_new_tokens=50,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

# ── QUESTION GENERATION ───────────────────────────────────
def get_question(q_num):
    history = ""
    for i in range(len(state["questions"])):
        history += f"\nQuestion {i+1}: {state['questions'][i]}"
        if i < len(state["answers"]):
            history += f"\nAnswer {i+1}: {state['answers'][i]}"
        if i < len(state["actions"]):
            history += f"\nAction {i+1}: {state['actions'][i]}"
        history += "\n"

    last_action = state["actions"][-1] if state["actions"] else "START"

    prompt = build_prompt(f"""
You are conducting a real technical interview.

Role: {ROLE}
Experience: {EXPERIENCE}
JD Skills: {', '.join(JD_SKILLS)}

Interview History:
{history}

Current Question Number: {q_num}
Previous Action: {last_action}

Rules:
1. Total interview questions = {NUM_QUESTIONS}
2. Never ask more than {NUM_QUESTIONS} questions.
3. FOLLOW_UP: Ask clarification on SAME topic.
4. DEEP_DIVE: Ask advanced production-level question on SAME topic.
5. NEXT_TOPIC: Move to a DIFFERENT topic.
6. If candidate clearly does not know the topic: Move to NEXT_TOPIC.
7. Never repeat question/topic/skill.
8. Ask only ONE question.
9. No score. No feedback. No action text.

Output: Question only.
""")
    return run_model(prompt, max_tok=120)

# ── FEEDBACK GENERATION ───────────────────────────────────
def get_feedback(question, answer):
    prompt = build_prompt(f"""
Question: {question}
Candidate Answer: {answer}

You are evaluating a technical interview answer.

Rules:
1. If candidate says "I don't know" / "no idea" / "never worked on this" / "not sure" / "skip" / "cannot answer":
   Feedback: Candidate does not know this topic.
   Score: 0/10
   Action: NEXT_TOPIC
2. Weak answer: Score 1-4
3. Partial answer: Score 5-7
4. Strong answer: Score 8-10
5. FOLLOW_UP: partial knowledge shown
6. DEEP_DIVE: strong knowledge shown
7. NEXT_TOPIC: no knowledge OR topic sufficiently covered

Output exactly:
Feedback: <2 sentences>
Score: X/10
Action: FOLLOW_UP or DEEP_DIVE or NEXT_TOPIC
""")
    return run_feedback_model(prompt, max_tok=150)

# ── FINAL REPORT (base model) ─────────────────────────────
def generate_final_report():
    qa_summary = ""
    for i, (q, a, sc, ac) in enumerate(zip(
        state["questions"], state["answers"], state["scores"], state["actions"]
    )):
        qa_summary += f"Q{i+1}: {q}\nAnswer: {a[:150]}\nScore: {sc}/10 | Action: {ac}\n\n"

    avg = round(sum(state["scores"]) / len(state["scores"]), 1) if state["scores"] else 0

    prompt = (
        f"<s>[INST] You are a senior technical recruiter writing a hiring report.\n\n"
        f"Role: {ROLE}\nExperience Required: {EXPERIENCE}\nCompany: {JD_COMPANY}\n"
        f"JD Skills: {', '.join(JD_SKILLS)}\nResume Skills: {', '.join(RESUME_SKILLS[:6])}\n\n"
        f"Interview Transcript:\n{qa_summary}\nAverage Score: {avg}/10\n\n"
        f"Write a hiring report in this EXACT format:\n\n"
        f"Overall Performance: [Excellent/Good/Average/Below Average/Poor]\n"
        f"Technical Score: {avg}/10\n"
        f"Communication: [Excellent/Good/Average/Poor]\n"
        f"Strengths: [2-3 specific strengths]\n"
        f"Weaknesses: [1-2 specific gaps]\n"
        f"JD Match: [High/Medium/Low] - [reason]\n"
        f"Hiring Decision: [Hire/Consider/Reject]\n"
        f"Reason: [2 sentences]\n[/INST]"
    )
    return run_base_model(prompt, max_tok=350)

# ── HELPERS ───────────────────────────────────────────────
def extract_score(text):
    m = re.search(r'Score:\s*(\d+)/10', text, re.IGNORECASE)
    return int(m.group(1)) if m else 0

def extract_action(text):
    t = text.upper()
    if "DEEP_DIVE" in t: return "DEEP_DIVE"
    if "FOLLOW_UP" in t: return "FOLLOW_UP"
    return "NEXT_TOPIC"

# ── Updated metadata generator — exact format jo chahiye ──
def compute_metadata():
    avg = round(sum(state["scores"]) / len(state["scores"]), 1) if state["scores"] else 0
    quality  = "Strong" if avg >= 7.5 else "Average" if avg >= 5.5 else "Weak"
    decision = "Hire"   if avg >= 7.5 else "Consider" if avg >= 5.5 else "Reject"

    metadata = {
        "role": ROLE,
        "experience": EXPERIENCE,
        "quality": quality,
        "jd_company": JD_COMPANY,
        "topics": list(set(state.get("topics_covered", []))) or ["General"],
        "resume_skills": RESUME_SKILLS,
        "resume_projects": RESUME_PROJECTS,
        "jd_skills": JD_SKILLS,
        "generation_seed": {
            "role": ROLE,
            "experience": EXPERIENCE,
            "quality": quality,
            "jd_company": JD_COMPANY
        },
        "expected_performance": quality,
        "target_hiring_decision": decision,
    }
    return avg, quality, decision, metadata

# ════════════════════════════════════════════════════════════
# MAIN LOOP — SIMPLE INPUT() BASED
# ════════════════════════════════════════════════════════════

print("="*65)
print(f"  🤖 AI INTERVIEWER | {ROLE} | {EXPERIENCE}")
print(f"  🏢 {JD_COMPANY} | ❓ {NUM_QUESTIONS} Questions")
print("="*65)

for q_num in range(1, NUM_QUESTIONS + 1):
    print(f"\n⏳ Generating question {q_num}...")
    question = get_question(q_num)
    state["questions"].append(question)

    print(f"\n{'─'*65}")
    print(f"❓ Question {q_num}/{NUM_QUESTIONS}:")
    print(question)
    print(f"{'─'*65}")

    # Take input
    answer = input("👤 Your Answer: ")
    state["answers"].append(answer)

    print("\n⏳ Evaluating...")
    fb = get_feedback(question, answer)
    score  = extract_score(fb)
    action = extract_action(fb)

    state["feedbacks"].append(fb)
    state["scores"].append(score)
    state["actions"].append(action)

    print(f"\n🎤 Feedback:")
    print(fb)
    print(f"{'─'*65}")




#################
print("\n" + "="*65)
print("⏳ Generating final hiring report (base model)...")
print("="*65)

avg, quality, decision, metadata = compute_metadata()
decision_icon = "✅" if decision == "Hire" else "⚠️" if decision == "Consider" else "❌"

print(f"\n🏁 INTERVIEW COMPLETE")
print(f"{decision_icon} Decision: {decision} | Avg Score: {avg}/10 | Quality: {quality}")
print("="*65)

print("\n📊 metadata:")
print(json.dumps({"metadata": metadata}, indent=2))